# Complexity-08b — Origami Wolfram pli 4 : les combinateurs S/K/I, ou la representation ultime du calcul (Rust)

**Navigation** : [<< Complexity-08 — La règle 30 comme générateur d'aléa, la règle 110 et son universalité](Complexity-08-WolframRule30Aleatoire-Rule110Universelle-Python.ipynb) | [Index de la série](README.md) | [Complexity-08c — Machines de Gödel →](Complexity-08c-MachinesDeGoedel-Python.ipynb) | [jumeau Python](Complexity-08b-CombinateursSKI-TagDePost-Python.ipynb) | [jumeau C#](Complexity-08b-CombinateursSKI-TagDePost-CSharp.ipynb)

**Carnet** : deux points de la definition du pli 4 qu'aucun carnet de la serie ne couvre encore --
la **logique combinatoire S/K/I** (syntaxe, reduction, forme normale, abstraction de bracket,
entiers de Church) et le **probleme de tag de Post** comme borne d'irreductibilite du geste
symbolique, avec, a cote de chaque these, sa **reception critique**.

**Organe utilise** : le jumeau Python importe `ict.combinators` (chemin
`MyIA.AI.Notebooks/IIT/ICT-Series/ict/combinators.py`). Toutes les reductions et evolutions de ce carnet
passent par un **moteur local en Rust standard**. Chaque cellule de code définit les fonctions dont elle a besoin.
Les **instruments de presentation** (tables de traces, comparaisons de tailles) restent distincts du moteur.

**Ce que ce carnet fait, et ce qu'il ne fait pas.**

| Ces affirmations | Statut | Ce que fait ce carnet |
|---|---|---|
| S et K suffisent a representer tout calcul | *theoreme* (Schonfinkel 1924 ; Church 1936) | il **demonstre le mecanisme sur des exemples** (derivation de I, entiers de Church, arithmetique) ; il ne re-prouve pas l'universalite |
| La reduction est confluente : toute forme normale est unique (Church-Rosser) | *theoreme* | il en donne le **temoin empirique** (ordres de reduction aleatoires -> une seule forme normale) ; la preuve formelle est donnée séparément en Lean |
| Le systeme de tag de Post (0->00, 1->1101, suppression de 3) s'arrete-t-il pour toute graine ? | *probleme ouvert* depuis 1921 | il **mesure** les comportements (arret, cycle, croissance) et les blocs inatteignables ; il ne tranche pas |

La distinction est le sujet d'une partie de ce carnet : un theoreme demontre sur exemples, un
temoin empirique et un probleme ouvert ne se citent pas de la meme facon.

> **Statut épistémique** — **Sans verdict à ce jour** : ce carnet n'a pas encore de ligne dans la [matrice de dissociations](../../../docs/ict/dissociations-matrix.md), donc aucune solidité n'est revendiquée ici. Ce qu'il mesure est reproductible depuis ses cellules ; ce n'est pas un acquis de la série.

## Sources primaires et regle de citation

Les textes utilises sont lus dans les copies archivees du gisement
`G:\Mon Drive\MyIA\IA\Bibliographie IA\` :

| Reference | Copie archivee |
|---|---|
| Wolfram, *Combinators: A Centennial View* (2020) | `Symbolic\Wolfram on Mathematics and Proof\2020 - Wolfram - Combinators - A Centennial View.html` |
| Wolfram, *After 100 Years, Can We Finally Crack Post's Problem of Tag?* (4 mars 2021) | `Automata\Wolfram Physics Project\2021 - Wolfram - After 100 Years, Can We Finally Crack Post's Problem of Tag.html` |
| Wolfram, *1920, 2020 and a $20,000 Prize: Announcing the S Combinator Challenge* (8 juin 2021) | `Symbolic\Wolfram on Mathematics and Proof\2021 - Wolfram - 1920-2020 and a $20000 Prize, Announcing the S Combinator Challenge.html` |
| Wolfram, *A Bibliography of Combinators* (arXiv 2106.11729, 2021) | `Symbolic\Wolfram on Mathematics and Proof\2021 - Wolfram - A Bibliography of Combinators (arXiv 2106.11729).pdf` |

**Convention de citation** : chaque affirmation attribuee a un texte cite la **section de la copie
archivee** (une capture HTML se cite par son titre de section, jamais par un numero de ligne ; le
PDF bibliographique par sa page). La reception critique (Curry, Hindley, Cardone, Bimbo, Klop)
est relevee dans la bibliographie archivee ci-dessus, pas dans les originaux -- et le carnet le
dit quand c'est le cas.

In [2]:
// Moteur natif : combinateurs S/K/I et tag de Post
// Le jumeau Python importe l'organe de la série IIT/ICT-Series, sans copie locale.
// Son installation éditable (`pip install -e MyIA.AI.Notebooks/IIT/ICT-Series`, cœur léger,
// sans PyPhi) dispense de modifier sys.path. Ici, ni Python ni installation ne sont requis.
// Le moteur est écrit en Rust standard ; les instruments de présentation restent distincts.
// Le formatage des entiers par Rust ne dépend pas des paramètres régionaux.
{
    // Ce bloc présente les fonctions du moteur ; les cellules suivantes sont autonomes.
    #[allow(dead_code)]
    mod moteur {
        use std::collections::{BTreeSet, HashSet};
        use std::rc::Rc;

        // Une application partage ses deux sous-termes sans recopier leurs arbres.
        type TR = Rc<Term>;
        enum Term {
            Var(String),
            App(TR, TR),
        }

        // Une feuille nomme un combinateur ou une variable libre (par exemple f ou x).
        fn var(nom: &str) -> TR {
            Rc::new(Term::Var(nom.to_owned()))
        }

        fn ap(f: TR, x: TR) -> TR {
            Rc::new(Term::App(f, x))
        }

        fn s_comb() -> TR { var("S") }
        fn k_comb() -> TR { var("K") }
        fn i_comb() -> TR { var("I") }

        // OMEGA ne se reproduit pas en un seul pas : la réduction se poursuit de redex en redex.
        fn omega() -> TR {
            let sii = ap(ap(s_comb(), i_comb()), i_comb());
            ap(Rc::clone(&sii), sii)
        }

        // Application associative à gauche ; les parenthèses délimitent un sous-terme.
        fn parse_ski(texte: &str) -> Option<TR> {
            fn groupe<'a>(jetons: &mut impl Iterator<Item = &'a str>, parenthese: bool) -> Option<TR> {
                let mut terme: Option<TR> = None;
                while let Some(jeton) = jetons.next() {
                    let atome = match jeton {
                        "(" => groupe(jetons, true)?,
                        ")" => return if parenthese { terme } else { None },
                        nom => var(nom),
                    };
                    terme = Some(match terme {
                        Some(f) => ap(f, atome),
                        None => atome,
                    });
                }
                if parenthese { None } else { terme }
            }

            let separe = texte.replace('(', " ( ").replace(')', " ) ");
            groupe(&mut separe.split_whitespace(), false)
        }

        // Seule une application en position d'argument a besoin de parenthèses.
        fn to_str(terme: &TR) -> String {
            fn ecrire(terme: &Term, sortie: &mut String) {
                match terme {
                    Term::Var(nom) => sortie.push_str(nom),
                    Term::App(f, x) => {
                        ecrire(f.as_ref(), sortie);
                        sortie.push(' ');
                        let groupe = matches!(x.as_ref(), Term::App(..));
                        if groupe { sortie.push('('); }
                        ecrire(x.as_ref(), sortie);
                        if groupe { sortie.push(')'); }
                    }
                }
            }
            let mut sortie = String::new();
            ecrire(terme.as_ref(), &mut sortie);
            sortie
        }

        fn est_var(terme: &TR, nom: &str) -> bool {
            matches!(terme.as_ref(), Term::Var(v) if v == nom)
        }

        // Reconnaître exactement l'arité de la règle à la racine.
        fn match_s(terme: &TR) -> Option<(&TR, &TR, &TR)> {
            let Term::App(sfg, x) = terme.as_ref() else { return None; };
            let Term::App(sf, g) = sfg.as_ref() else { return None; };
            let Term::App(s, f) = sf.as_ref() else { return None; };
            est_var(s, "S").then_some((f, g, x))
        }

        fn match_k(terme: &TR) -> Option<&TR> {
            let Term::App(kx, _) = terme.as_ref() else { return None; };
            let Term::App(k, x) = kx.as_ref() else { return None; };
            est_var(k, "K").then_some(x)
        }

        fn match_i(terme: &TR) -> Option<&TR> {
            let Term::App(i, x) = terme.as_ref() else { return None; };
            est_var(i, "I").then_some(x)
        }

        fn apply_root(terme: &TR) -> Option<TR> {
            if let Some((f, g, x)) = match_s(terme) {
                return Some(ap(ap(Rc::clone(f), Rc::clone(x)), ap(Rc::clone(g), Rc::clone(x))));
            }
            match_k(terme).or_else(|| match_i(terme)).map(Rc::clone)
        }

        // Ordre normal : racine, fonction, argument ; applicatif : fonction, argument, racine.
        fn step(terme: &TR, applicatif: bool) -> Option<TR> {
            if !applicatif {
                if let Some(suite) = apply_root(terme) {
                    return Some(suite);
                }
            }
            if let Term::App(f, x) = terme.as_ref() {
                if let Some(suite) = step(f, applicatif) {
                    return Some(ap(suite, Rc::clone(x)));
                }
                if let Some(suite) = step(x, applicatif) {
                    return Some(ap(Rc::clone(f), suite));
                }
            }
            if applicatif { apply_root(terme) } else { None }
        }

        // La trace, si demandée, inclut le terme initial ; None indique une borne épuisée.
        fn reduce_term(terme: &TR, applicatif: bool, borne: usize, tracer: bool)
            -> (Option<TR>, usize, Option<Vec<TR>>)
        {
            let mut courant = Rc::clone(terme);
            let mut trace = tracer.then(|| vec![Rc::clone(terme)]);
            for pas in 0..borne {
                let Some(suite) = step(&courant, applicatif) else {
                    return (Some(courant), pas, trace);
                };
                courant = suite;
                if let Some(etats) = trace.as_mut() {
                    etats.push(Rc::clone(&courant));
                }
            }
            (None, borne, trace)
        }

        fn contient(terme: &TR, nom: &str) -> bool {
            match terme.as_ref() {
                Term::Var(v) => v == nom,
                Term::App(f, x) => contient(f, nom) || contient(x, nom),
            }
        }

        // Abstraction de bracket sans optimisation B/C : tester d'abord l'absence de x.
        fn lam(x: &str, corps: &TR) -> TR {
            if !contient(corps, x) {
                return ap(k_comb(), Rc::clone(corps));
            }
            match corps.as_ref() {
                Term::Var(_) => i_comb(),
                Term::App(f, a) => ap(ap(s_comb(), lam(x, f)), lam(x, a)),
            }
        }

        fn church(n: usize) -> TR {
            let f = var("f");
            let mut corps = var("x");
            for _ in 0..n {
                corps = ap(Rc::clone(&f), corps);
            }
            lam("f", &lam("x", &corps))
        }

        // Appliquer le numéral à f et x, puis reconnaître f (f (... x)).
        fn church_to_int(terme: &TR) -> Option<usize> {
            let applique = ap(ap(Rc::clone(terme), var("f")), var("x"));
            let nf = reduce_term(&applique, false, 100_000, false).0?;
            let mut courant = &nf;
            let mut n = 0;
            while let Term::App(f, x) = courant.as_ref() {
                if !est_var(f, "f") { return None; }
                courant = x;
                n += 1;
            }
            est_var(courant, "x").then_some(n)
        }

        // Pour le tag binaire de Post, retirer trois symboles et ajouter le bloc
        // déterminé par le premier symbole retiré.
        fn tag_step(etat: &str) -> Option<String> {
            let suffixe = etat.get(3..)?;
            let bloc = match etat.as_bytes().first().copied()? {
                b'0' => "00",
                b'1' => "1101",
                _ => return None,
            };
            let mut suite = suffixe.to_owned();
            suite.push_str(bloc);
            Some(suite)
        }

        // Conserver l'état qui ferme le cycle ; « unbounded » signale seulement la borne.
        fn tag_evolve(graine: &str, borne: usize) -> (Vec<String>, &'static str) {
            let mut courant = graine.to_owned();
            let mut trace = vec![courant.clone()];
            let mut vus = HashSet::from([courant.clone()]);
            for _ in 0..borne {
                let Some(suite) = tag_step(&courant) else { return (trace, "halted"); };
                trace.push(suite.clone());
                if !vus.insert(suite.clone()) {
                    return (trace, "cycle");
                }
                courant = suite;
            }
            (trace, "unbounded")
        }

        // Balayer le produit lexicographique de l'alphabet observé dans cette seule trace.
        fn first_missing_blocks(trace: &[String], largeur: usize) -> Vec<String> {
            fn parcourir(alphabet: &BTreeSet<char>, reste: usize, bloc: &mut String,
                         vus: &HashSet<String>, absents: &mut Vec<String>) {
                if reste == 0 {
                    if !vus.contains(bloc.as_str()) {
                        absents.push(bloc.clone());
                    }
                    return;
                }
                for &c in alphabet {
                    bloc.push(c);
                    parcourir(alphabet, reste - 1, bloc, vus, absents);
                    bloc.pop();
                }
            }

            if largeur == 0 { return Vec::new(); }
            let mut alphabet = BTreeSet::new();
            let mut vus: HashSet<String> = HashSet::new();
            for etat in trace {
                let symboles: Vec<char> = etat.chars().collect();
                alphabet.extend(symboles.iter().copied());
                for bloc in symboles.windows(largeur) {
                    vus.insert(bloc.iter().copied().collect());
                }
            }
            let mut absents = Vec::new();
            parcourir(&alphabet, largeur, &mut String::new(), &vus, &mut absents);
            absents
        }
    }

    println!("Combinateurs (S/K/I + tag de Post) : port manuel natif en Rust");
    println!("Regles : S f g x = f x (g x) | K x y = x | I x = x");
};

Combinateurs (S/K/I + tag de Post) : port manuel natif en Rust


Regles : S f g x = f x (g x) | K x y = x | I x = x


## 1. Trois lettres : la syntaxe minimale du calcul

Le 7 decembre 1920, Moses Schonfinkel presente a Gottingen les combinateurs ; l'article de 11
pages qui suit (*Uber die Bausteine der mathematischen Logik*, 1924) introduit deux regles et
une seule operation -- l'application. Wolfram resume ce geste : parvenir a concevoir "that it's
even conceivable that everything can be defined in terms of them is a **supreme achievement of
a kind of abstract reductionism**" (*A Centennial View*, section *Combinators Arrive*).

Toute la syntaxe tient dans ce tableau :

| Combinateur | Regle de reduction | Lecture |
|---|---|---|
| `S f g x` | `-> f x (g x)` | distribue l'argument aux deux branches |
| `K x y` | `-> x` | selectionne et oublie |
| `I x` | `-> x` | identite -- **derivable** : `S K K x -> x` |

La cellule suivante reduit `S K K x` pas a pas avec l'organe : le combinateur I n'est pas un
primitif, c'est un theoreme de deux pas.

In [3]:
// Illustration 1 — Réduction de S K K x
{
    use std::rc::Rc;

    // Le moteur est local à cette cellule ; Rc partage les sous-termes inchangés.
    type TR = Rc<Term>;
    enum Term {
        Var(String),
        App(TR, TR),
    }

    fn ap(f: TR, x: TR) -> TR {
        Rc::new(Term::App(f, x))
    }

    // Accumuler les applications de gauche à droite à chaque niveau de parenthèses.
    fn parse_ski(texte: &str) -> Option<TR> {
        fn groupe<'a>(jetons: &mut impl Iterator<Item = &'a str>, parenthese: bool) -> Option<TR> {
            let mut terme: Option<TR> = None;
            while let Some(jeton) = jetons.next() {
                let atome = match jeton {
                    "(" => groupe(jetons, true)?,
                    ")" => return if parenthese { terme } else { None },
                    nom => Rc::new(Term::Var(nom.to_owned())),
                };
                terme = Some(match terme {
                    Some(f) => ap(f, atome),
                    None => atome,
                });
            }
            if parenthese { None } else { terme }
        }

        let separe = texte.replace('(', " ( ").replace(')', " ) ");
        groupe(&mut separe.split_whitespace(), false)
    }

    fn to_str(terme: &TR) -> String {
        match terme.as_ref() {
            Term::Var(nom) => nom.clone(),
            Term::App(f, x) => {
                let gauche = to_str(f);
                let droite = to_str(x);
                if matches!(x.as_ref(), Term::App(..)) {
                    format!("{gauche} ({droite})")
                } else {
                    format!("{gauche} {droite}")
                }
            }
        }
    }

    // Tester les règles S, K et I à la racine avant de parcourir les sous-termes.
    fn est_var(terme: &TR, nom: &str) -> bool {
        matches!(terme.as_ref(), Term::Var(v) if v == nom)
    }

    fn apply_root(terme: &TR) -> Option<TR> {
        let Term::App(f, x) = terme.as_ref() else { return None; };
        if let Term::App(h, g) = f.as_ref() {
            if let Term::App(s, sf) = h.as_ref() {
                if est_var(s, "S") {
                    return Some(ap(ap(Rc::clone(sf), Rc::clone(x)), ap(Rc::clone(g), Rc::clone(x))));
                }
            }
            if est_var(h, "K") {
                return Some(Rc::clone(g));
            }
        }
        est_var(f, "I").then(|| Rc::clone(x))
    }

    fn step(terme: &TR, applicatif: bool) -> Option<TR> {
        if !applicatif {
            if let Some(suite) = apply_root(terme) {
                return Some(suite);
            }
        }
        if let Term::App(f, x) = terme.as_ref() {
            if let Some(suite) = step(f, applicatif) {
                return Some(ap(suite, Rc::clone(x)));
            }
            if let Some(suite) = step(x, applicatif) {
                return Some(ap(Rc::clone(f), suite));
            }
        }
        if applicatif { apply_root(terme) } else { None }
    }

    // Compter les pas de réécriture, sans compter l'état initial comme un pas.
    fn reduce_term(terme: &TR, applicatif: bool, borne: usize, tracer: bool)
        -> (Option<TR>, usize, Option<Vec<TR>>)
    {
        let mut courant = Rc::clone(terme);
        let mut trace = tracer.then(|| vec![Rc::clone(terme)]);
        for pas in 0..borne {
            let Some(suite) = step(&courant, applicatif) else {
                return (Some(courant), pas, trace);
            };
            courant = suite;
            if let Some(etats) = trace.as_mut() {
                etats.push(Rc::clone(&courant));
            }
        }
        (None, borne, trace)
    }

    // Les témoins Python fixent une comparaison exacte, sans sortie de contrôle supplémentaire.
    const TRACE_ATTENDUE: [&str; 3] = ["S K K x", "K x (K x)", "x"];
    const NF_ATTENDUE: &str = "x";
    const PAS_ATTENDUS: usize = 2;
    let terme = parse_ski("S K K x");
    assert!(terme.is_some(), "Écart de parité : terme de départ non analysé");
    if let Some(terme) = terme {
        let (nf, pas, trace) = reduce_term(&terme, false, 100, true);
        let etats: Vec<String> = trace.into_iter().flatten().map(|t| to_str(&t)).collect();
        let forme = nf.as_ref().map(to_str);
        assert!(etats.iter().map(String::as_str).eq(TRACE_ATTENDUE), "Écart de parité : trace calculée {etats:?}");
        assert_eq!(pas, PAS_ATTENDUS, "Écart de parité : nombre de pas");
        assert_eq!(forme.as_deref(), Some(NF_ATTENDUE), "Écart de parité : forme normale");
        for (i, etat) in etats.iter().enumerate() {
            println!("pas {i}: {etat}");
        }
        if let Some(forme) = forme {
            println!("forme normale : {forme} | pas : {pas}");
        }
    }
};

pas 0: S K K x


pas 1: K x (K x)


pas 2: x


forme normale : x | pas : 2


**Lecture du resultat** : `S K K x -> K x (K x) -> x`. Deux lettres suffisent ; la troisieme
(I) s'obtient. C'est ce constat qui fonde le **S Combinator Challenge** (8 juin 2021, prix de
$20 000) : Wolfram y demande si **S seul** -- sans K -- est encore universel, decrivant Schonfinkel
comme ayant fourni "the first explicit example of a system capable of what we now call universal
computation" (section *Hiding in Plain Sight for a Century?*). La question du minimum absolu
reste ouverte un siecle apres : c'est le fil rouge de ce pli.

## 2. Ordre normal contre ordre applicatif : la strategie n'est pas cosmetique

Deux strategies classiques de reduction :

- **ordre normal** : reduire le redex le plus externe-gauche d'abord -- le theoreme de
  standardisation garantit qu'il trouve la forme normale quand elle existe ;
- **ordre applicatif** : evaluer les arguments d'abord -- la strategie des langages usuels, qui
  peut diverger sur un terme qui possede pourtant une forme normale.

Le temoin canonique est `K I OMEGA` (ou `OMEGA = S I I (S I I)` est le terme qui se reduit en
lui-meme, section 3) : la regle K n'a jamais besoin de son second argument.

In [4]:
// Illustration 2 — Deux stratégies sur K I OMEGA
{
    use std::rc::Rc;

    // Le moteur est local à la cellule ; les branches non réécrites sont partagées.
    type TR = Rc<Term>;
    enum Term {
        Var(String),
        App(TR, TR),
    }

    fn var(nom: &str) -> TR {
        Rc::new(Term::Var(nom.to_owned()))
    }

    fn ap(f: TR, x: TR) -> TR {
        Rc::new(Term::App(f, x))
    }

    fn k_comb() -> TR { var("K") }
    fn i_comb() -> TR { var("I") }

    fn omega() -> TR {
        let sii = ap(ap(var("S"), i_comb()), i_comb());
        ap(Rc::clone(&sii), sii)
    }

    fn to_str(terme: &TR) -> String {
        match terme.as_ref() {
            Term::Var(nom) => nom.clone(),
            Term::App(f, x) => {
                let gauche = to_str(f);
                let droite = to_str(x);
                if matches!(x.as_ref(), Term::App(..)) {
                    format!("{gauche} ({droite})")
                } else {
                    format!("{gauche} {droite}")
                }
            }
        }
    }

    fn est_var(terme: &TR, nom: &str) -> bool {
        matches!(terme.as_ref(), Term::Var(v) if v == nom)
    }

    // Les règles S, K et I sont identiques pour les deux stratégies.
    fn apply_root(terme: &TR) -> Option<TR> {
        let Term::App(f, x) = terme.as_ref() else { return None; };
        if let Term::App(h, g) = f.as_ref() {
            if let Term::App(s, sf) = h.as_ref() {
                if est_var(s, "S") {
                    return Some(ap(ap(Rc::clone(sf), Rc::clone(x)), ap(Rc::clone(g), Rc::clone(x))));
                }
            }
            if est_var(h, "K") {
                return Some(Rc::clone(g));
            }
        }
        est_var(f, "I").then(|| Rc::clone(x))
    }

    // Ordre normal : racine, fonction, argument ; applicatif : fonction, argument, racine.
    fn step(terme: &TR, applicatif: bool) -> Option<TR> {
        if !applicatif {
            if let Some(suite) = apply_root(terme) {
                return Some(suite);
            }
        }
        if let Term::App(f, x) = terme.as_ref() {
            if let Some(suite) = step(f, applicatif) {
                return Some(ap(suite, Rc::clone(x)));
            }
            if let Some(suite) = step(x, applicatif) {
                return Some(ap(Rc::clone(f), suite));
            }
        }
        if applicatif { apply_root(terme) } else { None }
    }

    // En l'absence de forme normale constatée dans le budget, renvoyer None.
    fn reduce_term(terme: &TR, applicatif: bool, borne: usize, tracer: bool)
        -> (Option<TR>, usize, Option<Vec<TR>>)
    {
        let mut courant = Rc::clone(terme);
        let mut trace = tracer.then(|| vec![Rc::clone(terme)]);
        for pas in 0..borne {
            let Some(suite) = step(&courant, applicatif) else {
                return (Some(courant), pas, trace);
            };
            courant = suite;
            if let Some(etats) = trace.as_mut() {
                etats.push(Rc::clone(&courant));
            }
        }
        (None, borne, trace)
    }

    // Témoins Python : comparaison exacte de la forme normale et des compteurs.
    const BORNE: usize = 500;
    const NORMAL_ATTENDU: (Option<&str>, usize) = (Some("I"), 1);
    const APPLICATIF_ATTENDU: (Option<&str>, usize) = (None, 500);
    let terme = ap(ap(k_comb(), i_comb()), omega());
    let (nf_n, pas_n, _) = reduce_term(&terme, false, BORNE, false);
    let (nf_a, pas_a, _) = reduce_term(&terme, true, BORNE, false);
    let normal = nf_n.as_ref().map(to_str);
    let applicatif = nf_a.as_ref().map(to_str);
    assert_eq!((normal.as_deref(), pas_n), NORMAL_ATTENDU, "Écart de parité : ordre normal");
    assert_eq!((applicatif.as_deref(), pas_a), APPLICATIF_ATTENDU, "Écart de parité : ordre applicatif");

    let texte_n = normal.as_deref().unwrap_or("diverge");
    let texte_a = applicatif.as_deref().unwrap_or("diverge");
    println!("ordre normal    : {texte_n} ({pas_n} pas)");
    println!("ordre applicatif: {texte_a} (borne {pas_a} pas atteinte)");
};

ordre normal    : I (1 pas)


ordre applicatif: diverge (borne 500 pas atteinte)


**Lecture du resultat** : le **meme terme**, la **meme regle K** -- et deux destins. En ordre
normal, `K I OMEGA -> I` en un pas : l'argument jamais utilise n'est jamais evalue. En ordre
applicatif, la machine s'enfonce dans OMEGA et ne revient pas. La paresse n'est pas une
optimisation : c'est une difference **semantique** -- certains calculs n'existent que si l'on
s'interdit d'en faire certains autres.

## 3. OMEGA : la divergence pure

Le terme `S I I (S I I)` est le point fixe de la reduction : chaque pas produit un terme qui se
reduit encore. Wolfram note que, contrairement aux automates cellulaires ou aux machines de
Turing, les combinateurs ont "a certain immediate structural complexity, with their elaborate
tree structures" (*A Centennial View*, section *What Is Their Mathematics?*) -- la divergence
elle-meme y est structurale, pas un simple non-terminus.

In [5]:
// Illustration 3 — OMEGA : la trace grossit, et aucune forme normale n'est atteinte.
{
    use std::rc::Rc;

    // Les nœuds immuables sont partagés : la trace conserve les états sans les recopier.
    type TR = Rc<Term>;
    enum Term {
        Var(String),
        App(TR, TR),
    }

    fn var(nom: &str) -> TR {
        Rc::new(Term::Var(nom.to_owned()))
    }

    fn ap(f: TR, x: TR) -> TR {
        Rc::new(Term::App(f, x))
    }

    fn omega() -> TR {
        let sii = ap(ap(var("S"), var("I")), var("I"));
        ap(Rc::clone(&sii), sii)
    }

    fn to_str(terme: &TR) -> String {
        match terme.as_ref() {
            Term::Var(nom) => nom.clone(),
            Term::App(f, x) => {
                let gauche = to_str(f);
                let droite = to_str(x);
                if matches!(x.as_ref(), Term::App(..)) {
                    format!("{gauche} ({droite})")
                } else {
                    format!("{gauche} {droite}")
                }
            }
        }
    }

    fn est_var(terme: &TR, nom: &str) -> bool {
        matches!(terme.as_ref(), Term::Var(v) if v == nom)
    }

    // Une seule contraction à la racine, si l'une des règles le permet.
    fn apply_root(terme: &TR) -> Option<TR> {
        let Term::App(f, x) = terme.as_ref() else { return None; };
        if let Term::App(h, g) = f.as_ref() {
            if let Term::App(s, sf) = h.as_ref() {
                if est_var(s, "S") {
                    return Some(ap(ap(Rc::clone(sf), Rc::clone(x)), ap(Rc::clone(g), Rc::clone(x))));
                }
            }
            if est_var(h, "K") {
                return Some(Rc::clone(g));
            }
        }
        est_var(f, "I").then(|| Rc::clone(x))
    }

    // Ordre normal : racine, fonction, argument ; applicatif : fonction, argument, racine.
    fn step(terme: &TR, applicatif: bool) -> Option<TR> {
        if !applicatif {
            if let Some(suite) = apply_root(terme) {
                return Some(suite);
            }
        }
        if let Term::App(f, x) = terme.as_ref() {
            if let Some(suite) = step(f, applicatif) {
                return Some(ap(suite, Rc::clone(x)));
            }
            if let Some(suite) = step(x, applicatif) {
                return Some(ap(Rc::clone(f), suite));
            }
        }
        if applicatif { apply_root(terme) } else { None }
    }

    fn reduce_term(terme: &TR, applicatif: bool, borne: usize, tracer: bool)
        -> (Option<TR>, usize, Option<Vec<TR>>)
    {
        let mut courant = Rc::clone(terme);
        let mut trace = tracer.then(|| vec![Rc::clone(terme)]);
        for pas in 0..borne {
            let Some(suite) = step(&courant, applicatif) else {
                return (Some(courant), pas, trace);
            };
            courant = suite;
            if let Some(etats) = trace.as_mut() {
                etats.push(Rc::clone(&courant));
            }
        }
        (None, borne, trace)
    }

    // Témoins Python : vérifier le terme initial et chaque état de la trace.
    const TRACE_ATTENDUE: [&str; 4] = [
        "S I I (S I I)",
        "I (S I I) (I (S I I))",
        "S I I (I (S I I))",
        "I (I (S I I)) (I (I (S I I)))",
    ];
    const PAS_TRACE: usize = 3;
    const BORNE: usize = 200;
    let terme = omega();
    let initial = to_str(&terme);
    let (nf_courte, pas_courts, trace) = reduce_term(&terme, false, PAS_TRACE, true);
    let etats: Vec<String> = trace.into_iter().flatten().map(|t| to_str(&t)).collect();
    assert_eq!(initial.as_str(), TRACE_ATTENDUE[0], "Écart de parité : terme initial");
    assert!(etats.iter().map(String::as_str).eq(TRACE_ATTENDUE), "Écart de parité : trace calculée {etats:?}");
    assert_eq!((nf_courte.is_none(), pas_courts), (true, PAS_TRACE), "Écart de parité : réduction courte");

    println!("terme initial : {initial}");
    for (i, etat) in etats.iter().enumerate() {
        println!("pas {i}: {etat}");
    }

    // « diverge » reprend le libellé Python : le test constate seulement
    // qu'aucune forme normale n'est atteinte en 200 pas. Une preuve de
    // divergence doit établir séparément que la réduction se poursuit indéfiniment.
    let (nf, pas, _) = reduce_term(&terme, false, BORNE, false);
    let verdict = nf.as_ref().map_or_else(|| "diverge -- aucune forme normale atteinte".to_owned(), to_str);
    // Contrôle borné : aucune forme normale atteinte en 200 pas d'ordre normal.
    assert_eq!((nf.is_none(), pas), (true, BORNE), "Écart de parité : réduction bornée d'OMEGA");
    println!("apres {pas} pas : {verdict}");
};

terme initial : S I I (S I I)


pas 0: S I I (S I I)


pas 1: I (S I I) (I (S I I))


pas 2: S I I (I (S I I))


pas 3: I (I (S I I)) (I (I (S I I)))


apres 200 pas : diverge -- aucune forme normale atteinte


**Lecture du resultat** : la trace grossit a chaque pas -- le terme se replie sur sa propre
structure sans jamais s'aplatir. C'est le materiau brut de l'indecidabilite : Church et Kleene
introduisent la lambda-definissabilite vers 1934, Turing les machines en 1936, et "that -- in
effect by thinking about having it compile itself -- Hilbert's Entscheidungsproblem couldn't be
solved" (*A Centennial View*, section *Godel's Theorem and Computability*). Le meme essay
rapporte que Church demontre l'indecidabilite de l'equivalence des expressions lambda avant que
Turing n'etablisse la correspondance avec ses machines (section *Lambda Calculus*). Un moteur
assez riche pour tout calculer est assez riche pour ne rien garantir.

## 4. L'abstraction de bracket : eliminer les variables

Le pont entre le lambda-calcul de Church et les combinateurs est l'**abstraction de bracket** :
traduire `\x. M` en S/K/I par trois schemas -- `[x.x] = I`, `[x.c] = K c` (c sans x),
`[x.(M N)] = S [x.M] [x.N]`. Wolfram releve que Church, "on the very first page he defines
lambda, he's referencing Schonfinkel's combinator paper" (*A Centennial View*, section *Lambda
Calculus*) : les deux formalismes sont nes enchevetres.

La fonction `lam` implemente la traduction **pure**, sans les optimisations de Curry (combinateurs
B, C...) : la croissance du terme traduit est justement l'objet de la mesure -- optimiser la
traduction, ce serait maquiller le prix de l'elimination des variables.

In [6]:
// Illustration 4 — Abstraction de bracket et taille des termes
{
    use std::rc::Rc;

    // Un terme est une feuille ou une application de deux sous-termes partagés.
    type TR = Rc<Term>;
    enum Term {
        Var(String),
        App(TR, TR),
    }

    fn var(nom: &str) -> TR {
        Rc::new(Term::Var(nom.to_owned()))
    }

    fn ap(f: TR, x: TR) -> TR {
        Rc::new(Term::App(f, x))
    }

    // Associer les applications à gauche ; les parenthèses ouvrent un sous-terme.
    fn parse_ski(texte: &str) -> Option<TR> {
        fn groupe<'a>(jetons: &mut impl Iterator<Item = &'a str>, parenthese: bool) -> Option<TR> {
            let mut terme: Option<TR> = None;
            while let Some(jeton) = jetons.next() {
                let atome = match jeton {
                    "(" => groupe(jetons, true)?,
                    ")" => return if parenthese { terme } else { None },
                    nom => var(nom),
                };
                terme = Some(match terme {
                    Some(f) => ap(f, atome),
                    None => atome,
                });
            }
            if parenthese { None } else { terme }
        }

        let separe = texte.replace('(', " ( ").replace(')', " ) ");
        groupe(&mut separe.split_whitespace(), false)
    }

    fn to_str(terme: &TR) -> String {
        match terme.as_ref() {
            Term::Var(nom) => nom.clone(),
            Term::App(f, x) => {
                let gauche = to_str(f);
                let droite = to_str(x);
                if matches!(x.as_ref(), Term::App(..)) {
                    format!("{gauche} ({droite})")
                } else {
                    format!("{gauche} {droite}")
                }
            }
        }
    }

    fn contient(terme: &TR, nom: &str) -> bool {
        match terme.as_ref() {
            Term::Var(v) => v == nom,
            Term::App(f, x) => contient(f, nom) || contient(x, nom),
        }
    }

    // Appliquer les trois schémas de bracket, sans les optimisations B/C.
    fn lam(x: &str, corps: &TR) -> TR {
        if !contient(corps, x) {
            return ap(var("K"), Rc::clone(corps));
        }
        match corps.as_ref() {
            Term::Var(_) => var("I"),
            Term::App(f, a) => ap(ap(var("S"), lam(x, f)), lam(x, a)),
        }
    }

    fn church(n: usize) -> TR {
        let f = var("f");
        let mut corps = var("x");
        for _ in 0..n {
            corps = ap(Rc::clone(&f), corps);
        }
        lam("f", &lam("x", &corps))
    }

    // Nombre de nœuds (Var ou App) du terme : compter chaque occurrence,
    // même si plusieurs occurrences partagent le même Rc en mémoire.
    fn taille(t: &TR) -> usize {
        match t.as_ref() {
            Term::Var(_) => 1,
            Term::App(f, x) => 1 + taille(f) + taille(x),
        }
    }

    // Témoins Python, avec comparaison exacte des traductions et des tailles.
    const TRADUCTIONS_ATTENDUES: [&str; 3] = ["I", "K c", "S (S (K f) I) (S (K g) I)"];
    const TAILLES_ATTENDUES: [usize; 5] = [19, 35, 51, 67, 83];
    let exemples = [
        (r"\x.x           ->", Some(var("x"))),
        (r"\x.c           ->", Some(var("c"))),
        (r"\x.f x (g x)   ->", parse_ski("f x (g x)")),
    ];
    for ((etiquette, corps), attendu) in exemples.into_iter().zip(TRADUCTIONS_ATTENDUES) {
        let traduction = corps.as_ref().map(|t| to_str(&lam("x", t)));
        assert_eq!(traduction.as_deref(), Some(attendu), "Écart de parité : {etiquette}");
        if let Some(traduction) = traduction {
            println!("{etiquette} {traduction}");
        }
    }

    for (n, attendu) in (1..=5).zip(TAILLES_ATTENDUES) {
        let noeuds = taille(&church(n));
        assert_eq!(noeuds, attendu, "Écart de parité : taille de church({n})");
        println!("church({n}) en S/K pur : {noeuds:4} noeuds");
    }
};

\x.x           -> I


\x.c           -> K c


\x.f x (g x)   -> S (S (K f) I) (S (K g) I)


church(1) en S/K pur :   19 noeuds


church(2) en S/K pur :   35 noeuds


church(3) en S/K pur :   51 noeuds


church(4) en S/K pur :   67 noeuds


church(5) en S/K pur :   83 noeuds


**Lecture du resultat** : `\x. f x (g x)` devient `S (S (K f) I) (S (K g) I)` -- chaque variable
eliminee laisse une trace de structure. La mesure des tailles est nette : `church(n)` coute
`16 n + 3` noeuds en traduction pure, une croissance **lineaire mais au facteur 16**. Le prix
de l'elimination des variables n'est pas un rhetoric : c'est un facteur mesurable, et les
optimisations de Curry (B, C...) sont precisement l'ingenierie qui l'a combattu historiquement.

## 5. Les entiers de Church : l'arithmetique sans nombres

Un entier n est la fonction qui applique n fois son argument : `church(n) = \f x. f^n x`.
L'addition, la multiplication et l'exponentiation sont alors des **compositions** : ajouter,
c'est enchainer ; multiplier, c'est iterer l'iteration ; elever a une puissance, c'est iterer
la multiplication. Le carnet construit ces combinateurs par abstraction de bracket puis les
execute -- l'arithmetique est reelle, pas recitee.

In [7]:
// Illustration 5 — Arithmétique sur les entiers de Church
// Arithmétique sur les entiers d'Église : n = \f x. f^n x, encodés en combinateurs
// SKI. Les trois définitions ci-dessous sont les fermetures canoniques du
// lambda-calcul -- plus = \m n f x. m f (n f x), mult = \m n f x. n (m f) x,
// exp = \m n. n m -- et le round-trip church_to_int vérifie que 2+3=5, 2*3=6 et
// 2^3=8 : seul le décodage final compte des entiers natifs.
{
    use std::rc::Rc;

    // Rc partage les sous-termes inchangés pendant les réécritures.
    type TR = Rc<Term>;
    enum Term {
        Var(String),
        App(TR, TR),
    }

    fn var(nom: &str) -> TR {
        Rc::new(Term::Var(nom.to_owned()))
    }

    fn ap(f: TR, x: TR) -> TR {
        Rc::new(Term::App(f, x))
    }

    // Le parseur associe à gauche ; les parenthèses groupent un sous-terme.
    fn parse_ski(texte: &str) -> Option<TR> {
        fn groupe<'a>(jetons: &mut impl Iterator<Item = &'a str>, parenthese: bool) -> Option<TR> {
            let mut terme: Option<TR> = None;
            while let Some(jeton) = jetons.next() {
                let atome = match jeton {
                    "(" => groupe(jetons, true)?,
                    ")" => return if parenthese { terme } else { None },
                    nom => var(nom),
                };
                terme = Some(match terme {
                    Some(f) => ap(f, atome),
                    None => atome,
                });
            }
            if parenthese { None } else { terme }
        }
        let separe = texte.replace('(', " ( ").replace(')', " ) ");
        groupe(&mut separe.split_whitespace(), false)
    }

    fn to_str(terme: &TR) -> String {
        match terme.as_ref() {
            Term::Var(nom) => nom.clone(),
            Term::App(f, x) => {
                let gauche = to_str(f);
                let droite = to_str(x);
                if matches!(x.as_ref(), Term::App(..)) {
                    format!("{gauche} ({droite})")
                } else {
                    format!("{gauche} {droite}")
                }
            }
        }
    }

    fn est_var(terme: &TR, nom: &str) -> bool {
        matches!(terme.as_ref(), Term::Var(v) if v == nom)
    }

    fn apply_root(terme: &TR) -> Option<TR> {
        let Term::App(f, x) = terme.as_ref() else { return None; };
        if let Term::App(h, g) = f.as_ref() {
            if let Term::App(s, sf) = h.as_ref() {
                if est_var(s, "S") {
                    return Some(ap(ap(Rc::clone(sf), Rc::clone(x)), ap(Rc::clone(g), Rc::clone(x))));
                }
            }
            if est_var(h, "K") {
                return Some(Rc::clone(g));
            }
        }
        est_var(f, "I").then(|| Rc::clone(x))
    }

    // Ordre normal : tenter la racine, puis la fonction, puis l'argument.
    fn step_normal(terme: &TR) -> Option<TR> {
        if let Some(suite) = apply_root(terme) {
            return Some(suite);
        }
        let Term::App(f, x) = terme.as_ref() else { return None; };
        if let Some(suite) = step_normal(f) {
            return Some(ap(suite, Rc::clone(x)));
        }
        step_normal(x).map(|suite| ap(Rc::clone(f), suite))
    }

    fn reduce_normal(mut terme: TR, borne: usize) -> Option<TR> {
        for _ in 0..borne {
            let Some(suite) = step_normal(&terme) else { return Some(terme); };
            terme = suite;
        }
        None
    }

    fn contient(terme: &TR, nom: &str) -> bool {
        match terme.as_ref() {
            Term::Var(v) => v == nom,
            Term::App(f, x) => contient(f, nom) || contient(x, nom),
        }
    }

    // Traduction pure, sans règles d'optimisation B/C.
    fn lam(x: &str, corps: &TR) -> TR {
        if !contient(corps, x) {
            return ap(var("K"), Rc::clone(corps));
        }
        match corps.as_ref() {
            Term::Var(_) => var("I"),
            Term::App(f, a) => ap(ap(var("S"), lam(x, f)), lam(x, a)),
        }
    }

    fn church(n: usize) -> TR {
        let f = var("f");
        let mut corps = var("x");
        for _ in 0..n {
            corps = ap(Rc::clone(&f), corps);
        }
        lam("f", &lam("x", &corps))
    }

    // Appliquer le terme à f et x ; ne compter que les formes f (f (... x)).
    fn church_to_int(terme: &TR) -> Option<usize> {
        let applique = ap(ap(Rc::clone(terme), var("f")), var("x"));
        let nf = reduce_normal(applique, 100_000)?;
        let mut courant = &nf;
        let mut entier = 0;
        while let Term::App(f, x) = courant.as_ref() {
            if !est_var(f, "f") { return None; }
            entier += 1;
            courant = x;
        }
        est_var(courant, "x").then_some(entier)
    }

    fn abstraire(variables: &[&str], corps: &str) -> Option<TR> {
        let mut terme = parse_ski(corps)?;
        for &nom in variables.iter().rev() {
            terme = lam(nom, &terme);
        }
        Some(terme)
    }

    // Témoins du Python : égalité exacte de la structure imprimée et des entiers.
    const CHURCH_2: &str = "S (S (K S) (S (K K) I)) (S (S (K S) (S (K K) I)) (K I))";
    const ALLER_RETOUR: [usize; 8] = [0, 1, 2, 3, 4, 5, 6, 7];
    const ARITHMETIQUE: [Option<usize>; 3] = [Some(5), Some(6), Some(8)];
    let deux = church(2);
    let trois = church(3);
    let ecriture = to_str(&deux);
    let aller_retour: Option<Vec<usize>> = (0..8).map(|n| church_to_int(&church(n))).collect();
    let plus = abstraire(&["m", "n", "f", "x"], "m f (n f x)");
    let mult = abstraire(&["m", "n", "f", "x"], "n (m f) x");
    let exp = abstraire(&["m", "n"], "n m");
    let calculer = |operation: Option<TR>| {
        let operation = operation?;
        let expression = ap(ap(operation, Rc::clone(&deux)), Rc::clone(&trois));
        church_to_int(&expression)
    };
    let resultats = [plus, mult, exp].map(calculer);
    assert_eq!(ecriture.as_str(), CHURCH_2, "Écart de parité : écriture de church(2)");
    assert_eq!(aller_retour.as_deref(), Some(ALLER_RETOUR.as_slice()), "Écart de parité : round-trip");
    assert_eq!(resultats, ARITHMETIQUE, "Écart de parité : addition, multiplication ou exponentiation");

    println!("church(2) = {ecriture}");
    if let Some(valeurs) = aller_retour {
        println!("round-trip 0..7 : {valeurs:?}");
    }
    if let [Some(somme), Some(produit), Some(puissance)] = resultats {
        println!("2 + 3 = {somme} | 2 * 3 = {produit} | 2 ^ 3 = {puissance}");
    }
};

church(2) = S (S (K S) (S (K K) I)) (S (S (K S) (S (K K) I)) (K I))


round-trip 0..7 : [0, 1, 2, 3, 4, 5, 6, 7]


2 + 3 = 5 | 2 * 3 = 6 | 2 ^ 3 = 8


**Lecture du resultat** : `5`, `6`, `8` -- calcules par `S` et `K` seuls, sans aucun entier
natif jusqu'a la conversion finale. C'est le sens operationnel de la these d'universalite :
non pas un slogan, mais le fait que trois lignes de composition remplacent les tables
d'operation. La lecture `n (m f)` pour la multiplication -- iterer `m` fois, puis iterer ce
resultat `n` fois -- est le moment exact ou l'exposant devient une iteration d'iterations.

## 6. Church-Rosser empirique : mille chemins, une seule forme normale

Le **theoreme de Church-Rosser** (confluence) etablit que si un terme a deux formes normales,
elles sont egales : l'ordre de reduction ne change pas le resultat, seulement le chemin. Ce
carnet n'en donne **pas une preuve** -- il en donne le **temoin empirique** : depuis un meme
terme, reduire un redex **aleatoire** a chaque pas, 30 fois, et compter les formes normales
distinctes.

In [8]:
// Illustration 6 — Trente réductions aléatoires à graine fixe
{
    use std::collections::BTreeSet;
    use std::rc::Rc;

    // Chaque contraction partage les branches qui ne changent pas.
    type TR = Rc<Term>;
    enum Term {
        Var(String),
        App(TR, TR),
    }

    fn var(nom: &str) -> TR {
        Rc::new(Term::Var(nom.to_owned()))
    }

    fn ap(f: TR, x: TR) -> TR {
        Rc::new(Term::App(f, x))
    }

    fn to_str(terme: &TR) -> String {
        match terme.as_ref() {
            Term::Var(nom) => nom.clone(),
            Term::App(f, x) => {
                let gauche = to_str(f);
                let droite = to_str(x);
                if matches!(x.as_ref(), Term::App(..)) {
                    format!("{gauche} ({droite})")
                } else {
                    format!("{gauche} {droite}")
                }
            }
        }
    }

    fn est_var(terme: &TR, nom: &str) -> bool {
        matches!(terme.as_ref(), Term::Var(v) if v == nom)
    }

    // Séparer la reconnaissance du redex de la construction du terme contracté.
    enum Redex<'a> {
        S(&'a TR, &'a TR, &'a TR),
        K(&'a TR),
        I(&'a TR),
    }

    fn redex_racine(terme: &TR) -> Option<Redex<'_>> {
        let Term::App(f, x) = terme.as_ref() else { return None; };
        if let Term::App(h, g) = f.as_ref() {
            if let Term::App(s, sf) = h.as_ref() {
                if est_var(s, "S") { return Some(Redex::S(sf, g, x)); }
            }
            if est_var(h, "K") { return Some(Redex::K(g)); }
        }
        est_var(f, "I").then_some(Redex::I(x))
    }

    fn apply_root(terme: &TR) -> Option<TR> {
        match redex_racine(terme)? {
            Redex::S(f, g, x) => Some(ap(ap(Rc::clone(f), Rc::clone(x)), ap(Rc::clone(g), Rc::clone(x)))),
            Redex::K(x) | Redex::I(x) => Some(Rc::clone(x)),
        }
    }

    fn contient(terme: &TR, nom: &str) -> bool {
        match terme.as_ref() {
            Term::Var(v) => v == nom,
            Term::App(f, x) => contient(f, nom) || contient(x, nom),
        }
    }

    fn lam(x: &str, corps: &TR) -> TR {
        if !contient(corps, x) {
            return ap(var("K"), Rc::clone(corps));
        }
        match corps.as_ref() {
            Term::Var(_) => var("I"),
            Term::App(f, a) => ap(ap(var("S"), lam(x, f)), lam(x, a)),
        }
    }

    fn church(n: usize) -> TR {
        let f = var("f");
        let mut corps = var("x");
        for _ in 0..n {
            corps = ap(Rc::clone(&f), corps);
        }
        lam("f", &lam("x", &corps))
    }

    #[derive(Clone, Copy)]
    enum Branche { Fonction, Argument }

    // Énumérer les chemins en préordre : racine, fonction, argument.
    // Deux occurrences d'un sous-terme partagé restent deux positions distinctes.
    fn all_redex_paths(terme: &TR, chemin: &mut Vec<Branche>, chemins: &mut Vec<Vec<Branche>>) {
        if redex_racine(terme).is_some() {
            chemins.push(chemin.clone());
        }
        if let Term::App(f, x) = terme.as_ref() {
            chemin.push(Branche::Fonction);
            all_redex_paths(f, chemin, chemins);
            chemin.pop();
            chemin.push(Branche::Argument);
            all_redex_paths(x, chemin, chemins);
            chemin.pop();
        }
    }

    // Reconstruire seulement les applications qui mènent au redex choisi.
    fn replace_path(terme: &TR, chemin: &[Branche]) -> Option<TR> {
        let Some((branche, reste)) = chemin.split_first() else { return apply_root(terme); };
        let Term::App(f, x) = terme.as_ref() else { return None; };
        match branche {
            Branche::Fonction => Some(ap(replace_path(f, reste)?, Rc::clone(x))),
            Branche::Argument => Some(ap(Rc::clone(f), replace_path(x, reste)?)),
        }
    }

    // Générateur congruentiel modulo 2^64, distinct de celui de Python.
    // Utiliser les 32 bits de poids fort et rejeter la tranche excédentaire avant le modulo.
    struct Lcg(u64);
    impl Lcg {
        fn indice(&mut self, longueur: usize) -> Option<usize> {
            let n = u32::try_from(longueur).ok()?;
            if n == 0 { return None; }
            let seuil = n.wrapping_neg() % n;
            loop {
                self.0 = self.0.wrapping_mul(6_364_136_223_846_793_005).wrapping_add(1_442_695_040_888_963_407);
                let tirage = (self.0 >> 32) as u32;
                if tirage >= seuil { return Some((tirage % n) as usize); }
            }
        }
    }

    fn random_reduce(terme: &TR, rng: &mut Lcg, borne: usize) -> (Option<TR>, usize) {
        let mut courant = Rc::clone(terme);
        let mut chemin = Vec::new();
        let mut chemins = Vec::new();
        for pas in 0..borne {
            chemins.clear();
            all_redex_paths(&courant, &mut chemin, &mut chemins);
            if chemins.is_empty() { return (Some(courant), pas); }
            let Some(indice) = rng.indice(chemins.len()) else { return (None, pas); };
            let Some(suite) = replace_path(&courant, &chemins[indice]) else { return (None, pas); };
            courant = suite;
        }
        (None, borne)
    }

    // Témoins Python : comparer exactement la forme et les compteurs, pas les chemins tirés.
    const ESSAIS: usize = 30;
    const BORNE: usize = 20_000;
    const FORMES_ATTENDUES: [&str; 1] = ["f (f (f (f (f x))))"];
    const STATISTIQUES_ATTENDUES: (usize, usize, usize) = (42, 42, 42);
    let mut rng = Lcg(42); // Graine fixée : le témoin est reproductible.
    let terme = ap(ap(church(5), var("f")), var("x"));
    let mut formes = BTreeSet::new();
    let mut pas_vus = Vec::with_capacity(ESSAIS);
    for essai in 0..ESSAIS {
        let (nf, pas) = random_reduce(&terme, &mut rng, BORNE);
        assert!(nf.is_some(), "Écart de parité : pas de forme normale pour l'essai {}", essai + 1);
        if let Some(nf) = nf {
            formes.insert(to_str(&nf));
        }
        pas_vus.push(pas);
    }
    pas_vus.sort_unstable();
    // Même convention que sorted(pas_vus)[15] : prendre la valeur centrale supérieure.
    let min = pas_vus[0];
    let max = pas_vus[pas_vus.len() - 1];
    let median = pas_vus[ESSAIS / 2];
    assert!(formes.iter().map(String::as_str).eq(FORMES_ATTENDUES), "Écart de parité : formes obtenues {formes:?}");
    assert_eq!((min, max, median), STATISTIQUES_ATTENDUES, "Écart de parité : min, max ou médiane");

    println!("formes normales distinctes : {}", formes.len());
    for forme in &formes {
        println!("    {forme}");
    }
    println!("pas de reduction : min {min} | max {max} | median {median}");
};

formes normales distinctes : 1


    f (f (f (f (f x))))


pas de reduction : min 42 | max 42 | median 42


**Lecture du resultat -- et sa limite, dite**. Les 30 ordres convergent vers la meme forme
normale `f (f (f (f (f x))))` : c'est le temoin. Sur ce terme precis, le nombre de pas est meme
constant (42) : aucun redex du terme n'en duplique ni n'en detruit un autre -- une propriete de
ce terme, pas de la reduction en general (le redex d'OMEGA, lui, se duplique a chaque pas).

**Un temoin n'est pas une preuve** : trente ordres ne couvrent pas l'infinite des chemins. La
preuve formelle existe desormais : le lake
[`combinators_lean`](../../SymbolicAI/Lean/combinators_lean/) (pli 4, grain B) demontre
`confluence` -- toute paire de reductions `step*` issues d'un meme terme se rejoint -- et son
corollaire `unique_normal_form` : deux reductions exhaustives d'un meme terme aboutissent au
meme terme irreductible. La preuve invoque l'organe Mathlib `Relation.church_rosser` (forme de
Hindley-Rosen : le losange local de la reduction parallele entraine la confluence de sa cloture
reflexive-transitive), menee de style Tait-Martin-Lof sans lieurs -- ce que S/K/I rend bien plus
simple que le lambda-calcul. Ce carnet fournit l'instrument de mesure ; le lake fournit la
preuve verifiee par le noyau Lean.

**Note de parité** : le générateur congruentiel Rust, initialisé à 42, ne reproduit pas les tirages de `random.Random(42)`. Les 30 réductions conservent la même borne de 20 000 pas par essai. Le contrôle exige exactement la même forme normale et les mêmes minimum, maximum et médiane que dans le jumeau Python.

## 7. Le probleme de tag de Post : la borne d'irreductibilite

En 1920-1921, Emil Post simplifie la logique de Principia Mathematica en transformations de
chaines, jusqu'a un systeme minimal : alphabet `{0, 1}`, a chaque pas **supprimer 3 elements
en tete** puis **ajouter en queue** le bloc associe au premier element supprime (`0 -> 00`,
`1 -> 1101`). La question : **toute graine mene-t-elle a l'arret** (chaine plus courte que 3),
a un cycle, ou a une croissance sans borne ?

Wolfram raconte que Post "began on it in the early summer of 1921, no doubt expecting that such
a simple-to-state problem would have a correspondingly simple solution" (section *[Despite]
Considerable Effort... [It Proved] Intractable*) -- et que lui-meme doit reporter, un siecle
plus tard : "even a century later I still haven't been able to answer this question" (section
*So Does It Always Halt?*).

In [9]:
// Illustration 7 — Quatre trajectoires du tag de Post
{
    use std::collections::HashSet;

    // Sur cet alphabet binaire, un symbole occupe un octet.
    // Supprimer trois symboles, puis ajouter le bloc choisi par la tête retirée.
    fn tag_step(etat: &str) -> Option<String> {
        let suffixe = etat.get(3..)?;
        let bloc = match etat.as_bytes().first().copied()? {
            b'0' => "00",
            b'1' => "1101",
            _ => return None,
        };
        let mut suite = suffixe.to_owned();
        suite.push_str(bloc);
        Some(suite)
    }

    // Le vecteur conserve l'ordre des états, y compris la répétition qui ferme le cycle.
    // Le HashSet ne sert qu'au test d'appartenance, jamais à l'affichage.
    fn tag_evolve(graine: &str, borne: usize) -> (Vec<String>, &'static str) {
        let mut courant = graine.to_owned();
        let mut trace = vec![courant.clone()];
        let mut vus = HashSet::from([courant.clone()]);
        for _ in 0..borne {
            let Some(suite) = tag_step(&courant) else { return (trace, "halted"); };
            trace.push(suite.clone());
            if !vus.insert(suite.clone()) { return (trace, "cycle"); }
            courant = suite;
        }
        // Ce verdict indique une borne épuisée, pas une preuve de croissance sans borne.
        (trace, "unbounded")
    }

    // Représentation Python adaptée aux chaînes binaires de cette expérience.
    fn liste_python(etats: &[String]) -> String {
        let elements: Vec<String> = etats.iter().map(|etat| format!("'{etat}'")).collect();
        format!("[{}]", elements.join(", "))
    }

    // Témoins exacts du Python : verdict, effectif, longueurs, tête et fin de trace.
    type Temoin = (&'static str, &'static str, usize, usize, usize, &'static [&'static str], &'static [&'static str]);
    const TEMOINS: [Temoin; 3] = [
        ("10", "halted", 1, 2, 2, &["10"], &["10"]),
        ("1001010010000", "cycle", 33, 8, 16,
            &["1001010010000", "10100100001101", "001000011011101", "00001101110100"],
            &["0000110111011101", "011011101110100"]),
        ("110101", "cycle", 22, 6, 16,
            &["110101", "1011101", "11011101", "111011101"],
            &["0000110111011101", "011011101110100"]),
    ];
    const GRAINE_PERIODIQUE: &str = "1001001001001001001001001001001";
    const PERIODIQUE_ATTENDU: (usize, &str, usize) = (31, "cycle", 208);
    for (graine, verdict_attendu, nombre_attendu, min_attendu, max_attendu, tete_attendue, fin_attendue) in TEMOINS {
        let (trace, verdict) = tag_evolve(graine, 100_000);
        let nombre = trace.len();
        let (min, max) = trace.iter().map(String::len).fold(
            (graine.len(), graine.len()),
            |(min, max), longueur| (min.min(longueur), max.max(longueur)),
        );
        let tete = &trace[..nombre.min(4)];
        let fin = &trace[nombre.saturating_sub(2)..];
        assert_eq!((verdict, nombre, min, max), (verdict_attendu, nombre_attendu, min_attendu, max_attendu), "Écart de parité : résumé de la graine {graine}");
        assert!(tete.iter().map(String::as_str).eq(tete_attendue.iter().copied()), "Écart de parité : tête de la graine {graine}, calculée {tete:?}");
        assert!(fin.iter().map(String::as_str).eq(fin_attendue.iter().copied()), "Écart de parité : fin de la graine {graine}, calculée {fin:?}");

        let repr = format!("'{graine}'");
        println!("graine {repr:<18} -> {verdict:<9} | etats: {nombre:>5} | longueurs min/max: {min}/{max}");
        println!("   tete : {}", liste_python(tete));
        println!("   fin  : {}", liste_python(fin));
    }

    let (trace_longue, verdict_long) = tag_evolve(GRAINE_PERIODIQUE, 5_000);
    let longueur_graine = GRAINE_PERIODIQUE.len();
    let max_long = trace_longue.iter().map(String::len).max().unwrap_or(0);
    assert_eq!((longueur_graine, verdict_long, max_long), PERIODIQUE_ATTENDU, "Écart de parité : graine périodique");
    println!("graine periodique ({longueur_graine} elements) -> {verdict_long} | longueur max atteinte : {max_long}");
};

graine '10'               -> halted    | etats:     1 | longueurs min/max: 2/2


   tete : ['10']


   fin  : ['10']


graine '1001010010000'    -> cycle     | etats:    33 | longueurs min/max: 8/16


   tete : ['1001010010000', '10100100001101', '001000011011101', '00001101110100']


   fin  : ['0000110111011101', '011011101110100']


graine '110101'           -> cycle     | etats:    22 | longueurs min/max: 6/16


   tete : ['110101', '1011101', '11011101', '111011101']


   fin  : ['0000110111011101', '011011101110100']


graine periodique (31 elements) -> cycle | longueur max atteinte : 208


**Lecture du resultat** : trois destins mesures -- **arret** immediat (`10` est plus courte que
le seuil), **cycle** (la graine de 13 elements de l'essai de Wolfram, et une graine de 6,
rejoignent le meme attracteur), **cycle apres croissance** (une graine periodique de 31 monte
jusqu'a 208 elements avant de retomber). Aucune graine testee ici ne croit sans borne -- mais
l'echantillon ne prouve rien : c'est exactement le statut du probleme.

Post lui-meme voyait plus loin que son echec : "I would have discovered Godel's theorem in
1921 -- if I had been Godel" (cite par Wolfram, meme section introductive). Et en 1961, Marvin
Minsky demontre qu'un systeme de tag **convenablement construit** peut calculer tout ce qu'une
machine de Turing calcule (section *[Despite] Considerable Effort...*) : le probleme de Post
n'est pas dur parce que le systeme est pauvre -- il est dur parce qu'il est deja riche.

## 8. Blocs inatteignables et r = 1 : ou passe la frontiere causale

Deux faits de structure, tous deux mesures par Wolfram (2021) :

1. **Le bloc `1111` est inatteignable** : "no sequence of the 00s and 1101s inserted by the rule
   can ever produce it" (section *The Basic Setup*). Une preuve de deux lignes sur les regles --
   mais une contrainte permanente sur tout l'espace d'etats.
2. **La suppression d'un seul element par pas (r = 1) tue la causalite** : "rules which delete
   only one element at each step cannot involve real communication (or causal connections)
   between different parts of the string, and must be equivalent to neighbor-independent
   substitution systems" -- les regles `0 -> 01, 1 -> 10` engendrent le mot de Thue-Morse,
   `0 -> 01, 1 -> 0` le mot de Fibonacci (section *Other Tag Systems*).

In [10]:
// Illustration 8 — Blocs absents et substitutions à r = 1
{
    use std::collections::{BTreeSet, HashSet, VecDeque};

    // Tag binaire de Post : supprimer trois symboles, puis ajouter le bloc
    // déterminé par le premier symbole retiré.
    fn tag_step(etat: &str) -> Option<String> {
        let suffixe = etat.get(3..)?;
        let bloc = match etat.as_bytes().first().copied()? {
            b'0' => "00",
            b'1' => "1101",
            _ => return None,
        };
        let mut suite = suffixe.to_owned();
        suite.push_str(bloc);
        Some(suite)
    }

    // Conserver l'ordre de visite et l'état qui ferme le cycle.
    fn tag_evolve(graine: &str, borne: usize) -> (Vec<String>, &'static str) {
        let mut courant = graine.to_owned();
        let mut trace = vec![courant.clone()];
        let mut vus = HashSet::from([courant.clone()]);
        for _ in 0..borne {
            let Some(suite) = tag_step(&courant) else { return (trace, "halted"); };
            trace.push(suite.clone());
            if !vus.insert(suite.clone()) {
                return (trace, "cycle");
            }
            courant = suite;
        }
        (trace, "unbounded")
    }

    // Construire l'alphabet et les blocs observés dans cette trace seulement.
    fn first_missing_blocks(trace: &[String], largeur: usize) -> Vec<String> {
        fn parcourir(alphabet: &BTreeSet<char>, reste: usize, bloc: &mut String,
                     vus: &HashSet<String>, absents: &mut Vec<String>) {
            if reste == 0 {
                if !vus.contains(bloc.as_str()) {
                    absents.push(bloc.clone());
                }
                return;
            }
            // Parcourir l'alphabet trié à chaque niveau donne le produit lexicographique.
            for &c in alphabet {
                bloc.push(c);
                parcourir(alphabet, reste - 1, bloc, vus, absents);
                bloc.pop();
            }
        }

        if largeur == 0 { return Vec::new(); }
        let mut alphabet = BTreeSet::new();
        let mut vus: HashSet<String> = HashSet::new();
        for etat in trace {
            let symboles: Vec<char> = etat.chars().collect();
            alphabet.extend(symboles.iter().copied());
            for bloc in symboles.windows(largeur) {
                vus.insert(bloc.iter().copied().collect());
            }
        }
        let mut absents = Vec::new();
        parcourir(&alphabet, largeur, &mut String::new(), &vus, &mut absents);
        absents
    }

    fn liste_python(etats: &[String]) -> String {
        let elements: Vec<String> = etats.iter().map(|etat| format!("'{etat}'")).collect();
        format!("[{}]", elements.join(", "))
    }

    // Pour les entrées binaires, les deux cases correspondent aux symboles 0 et 1.
    type Regles = [&'static str; 2];
    const THUE_MORSE: Regles = ["01", "10"];
    const FIBONACCI: Regles = ["01", "0"];

    // Une génération d'un système de substitution : remplacer tous les symboles du mot.
    fn substitue(mot: &str, regles: Regles) -> String {
        let mut suivant = String::new();
        for symbole in mot.bytes() {
            suivant.push_str(regles[usize::from(symbole - b'0')]);
        }
        suivant
    }

    // Système de tag à r = 1 : chaque élément, à son tour, est remplacé par son bloc.
    fn evolue_r1(graine: &str, regles: Regles, n_pas: usize) -> Vec<String> {
        let mut etat: VecDeque<u8> = graine.bytes().collect();
        let mut etats = vec![graine.to_owned()];
        for _ in 0..n_pas {
            let Some(tete) = etat.pop_front() else { break; };
            etat.extend(regles[usize::from(tete - b'0')].bytes());
            etats.push(etat.iter().copied().map(char::from).collect());
        }
        etats
    }

    fn compte_prefixes_et_facteurs(etats: &[String], mot: &str) -> (usize, usize) {
        let prefixes = etats.iter().filter(|etat| mot.starts_with(etat.as_str())).count();
        let facteurs = etats.iter().filter(|etat| mot.contains(etat.as_str())).count();
        (prefixes, facteurs)
    }

    // Témoins du Python : comparaison exacte des listes, des mots et des entiers.
    const LARGEUR: usize = 4;
    const MANQUANTS_ATTENDUS: [&str; 2] = ["1100", "1111"];
    const THUE_ATTENDU: (usize, &str) = (64, "0110100110010110100101100110100110010110011010010110100110010110");
    const FIBO_ATTENDU: (usize, &str) = (34, "0100101001001010010100100101001001");
    const ETATS_ATTENDUS: (usize, usize) = (21, 21);
    const COMPTES_ATTENDUS: ((usize, usize), (usize, usize)) = ((5, 21), (6, 21));

    let (trace, _) = tag_evolve("1001010010000", 100_000);
    let manquants = first_missing_blocks(&trace, LARGEUR);
    let absent_1111 = manquants.iter().any(|bloc| bloc == "1111");
    assert!(manquants.iter().map(String::as_str).eq(MANQUANTS_ATTENDUS), "Écart de parité : blocs absents {manquants:?}");
    assert!(absent_1111, "Écart de parité : 1111 doit être absent de la trace");
    println!("blocs de largeur {LARGEUR} absents de la trace : {}", liste_python(&manquants));
    println!("1111 absent : {}", if absent_1111 { "True" } else { "False" });

    let mut thue = String::from("0");
    for _ in 0..6 {
        thue = substitue(&thue, THUE_MORSE);
    }
    let mut fibo = String::from("0");
    for _ in 0..7 {
        fibo = substitue(&fibo, FIBONACCI);
    }
    let taille_thue = thue.len();
    let taille_fibo = fibo.len();
    assert_eq!((taille_thue, thue.as_str()), THUE_ATTENDU, "Écart de parité : mot de Thue-Morse");
    assert_eq!((taille_fibo, fibo.as_str()), FIBO_ATTENDU, "Écart de parité : mot de Fibonacci");
    println!("Thue-Morse ({taille_thue}) : {thue}");
    println!("Fibonacci ({taille_fibo})  : {fibo}");

    // Les longueurs affichees sont mesurees, pas recopiees : un changement du nombre de
    // generations ne peut plus desynchroniser l'etiquette du mot qu'elle annonce.
    // Un etat de 21 elements ne se lit pas forcement dans le prefixe affiche (64) : les
    // facteurs se mesurent sur un prefixe plus long que la fenetre d'affichage.
    let mut thue_long = thue;
    for _ in 0..4 {
        thue_long = substitue(&thue_long, THUE_MORSE);
    }
    let mut fibo_long = fibo;
    for _ in 0..6 {
        fibo_long = substitue(&fibo_long, FIBONACCI);
    }

    let etats_tm = evolue_r1("0", THUE_MORSE, 20);
    let etats_fibo = evolue_r1("0", FIBONACCI, 20);
    let (prefixes_tm, facteurs_tm) = compte_prefixes_et_facteurs(&etats_tm, &thue_long);
    let (prefixes_fibo, facteurs_fibo) = compte_prefixes_et_facteurs(&etats_fibo, &fibo_long);
    let nombre_etats = etats_tm.len();
    assert_eq!((nombre_etats, etats_fibo.len()), ETATS_ATTENDUS, "Écart de parité : effectifs des traces à r = 1");
    assert_eq!(((prefixes_tm, facteurs_tm), (prefixes_fibo, facteurs_fibo)), COMPTES_ATTENDUS,
               "Écart de parité : décomptes des préfixes et facteurs");
    println!("sur {nombre_etats} etats r=1 :");
    println!("  prefixes de Thue-Morse : {prefixes_tm}");
    println!("  facteurs de Thue-Morse : {facteurs_tm}");
    println!("  prefixes de Fibonacci  : {prefixes_fibo}");
    println!("  facteurs de Fibonacci  : {facteurs_fibo}");
};

blocs de largeur 4 absents de la trace : ['1100', '1111']


1111 absent : True


Thue-Morse (64) : 0110100110010110100101100110100110010110011010010110100110010110


Fibonacci (34)  : 0100101001001010010100100101001001


sur 21 etats r=1 :


  prefixes de Thue-Morse : 5


  facteurs de Thue-Morse : 21


  prefixes de Fibonacci  : 6


  facteurs de Fibonacci  : 21


**Lecture du resultat** : a `r = 1`, les etats visites sont des **facteurs** (sous-mots) des
mots de Thue-Morse et de Fibonacci -- des suites parfaitement regulieres, generees par
substitution, sans aucune communication entre parties de la chaine. Ils ne sont pas
**prefixes** pour autant : 5 des 21 etats seulement sont des prefixes du mot de Thue-Morse, 6
pour Fibonacci ; les autres demarrent plus loin dans le mot. Le systeme de Post, lui, supprime
**trois** elements par pas : ce que la tete supprime depend de ce que la queue a recu, la
boucle causale est fermee, et la trace y laisse des trous -- `1111` est absent de la trace de
la graine de reference, et aucun bloc ajoute (`00` ou `1101`) ne peut le fabriquer, faute de
quatre `1` consecutifs ou de deux `1` terminaux. Une graine arbitraire, elle, en produit : les
496 graines de longueur 4 a 8 sont 121 a en faire apparaitre, et c'est l'exercice 3 qui le
mesure. La frontiere entre le systeme de substitution (compris) et le tag (ouvert) passe la --
entre `r = 1` et `r = 2`, un seul element supprime de plus.

## 9. Reception critique : un siecle de combinateurs, lue depuis 2021

La these wolframienne -- Schonfinkel fournit "the first explicit example of a system capable of
what we now call universal computation" (*S Combinator Challenge*, section *Hiding in Plain
Sight for a Century?*) -- se confronte a une tradition que Wolfram lui-meme a patiemment
relevee : *A Bibliography of Combinators* (arXiv 2106.11729) recense **plus de 200 entrees**
sur un siecle, de Curry (a partir de 1927-1930) aux travaux modernes de term rewriting.

| Volet de la reception | Reference | Ce qu'elle oppose a la these |
|---|---|---|
| Systematisation | Curry, Hindley & Seldin, *Combinatory Logic* vol. II (1972) | le programme n'est pas une decouverte isolee de 1920 : c'est quarante ans de construction de Curry, achevee en 1972 |
| Synthese moderne | Hindley & Seldin, *Lambda-Calculus and Combinators: an Introduction* (1980/2008) | le lambda-calcul et les combinateurs forment un seul corpus enseignable, pas un curiosity isole |
| Histoire | Cardone & Hindley, *History of Lambda-calculus and Combinatory Logic* (2006/2009) | les combinateurs se comprennent dans la continuite Frege-Principia-Church, pas comme un big bang |
| Acces canonique | Bimbo, *Combinatory Logic*, Stanford Encyclopedia of Philosophy (2009) | la communaute logique a son article de reference -- la place des combinateurs est institutionnalisee |
| Comportement computationnel | Klop, van Oostrom & van Raamsdonk, *Combinatory term rewriting systems* (TCS 121, 1993) | la theorie du rewriting a traite les combinateurs comme objet d'etude a part entiere, bien avant la ruliologie |

Ces references sont relevees dans la **bibliographie archivee de Wolfram lui-meme** (PDF
arXiv 2106.11729) -- le carnet cite la grille de lecture wolframienne et ses propres sources,
pas les originaux : c'est une reception de reception, et il le dit.

Le point de friction le plus net : la these de la **representation ultime**. Wolfram ecrit les
combinateurs comme le fondement le plus simple du calcul ; la tradition Curry-Hindley les traite
comme un **formalisme commode** -- une notation sans variables pour raisonner sur les fonctions
et les modeles de la logique combinatoire. Les deux lectures ne se contredisent pas ; elles ne
parlent pas de la meme question. Le S Combinator Challenge ($20 000, 2021) est la seule piece
qui les force a se rejoindre : si S seul est universel, la these du minimum absolu se renforce
de l'interieur du formalisme -- et le challenge est, a ce jour, non resolu.

## 10. Synthese -- trois statuts, un seul instrument

| Affirmation | Source | Statut | Ce que ce carnet a fait |
|---|---|---|---|
| S K K x -> x : I est derivable | Schonfinkel 1924 | theoreme trivial | reduit pas a pas (section 1) |
| L'ordre de reduction change l'issu : K I OMEGA | standardisation | theoreme | les deux ordres executes (section 2) |
| S/K represente l'arithmetique | Church 1936 | theoreme | 2+3, 2*3, 2^3 calcules (section 5) |
| Toute forme normale est unique | Church-Rosser | theoreme | **temoin empirique** : 30 ordres -> 1 forme (section 6) ; preuve formelle = grain B (Lean) |
| 1111 est inatteignable dans le tag de Post | Wolfram 2021, *The Basic Setup* | fait prouvable sur les regles | mesure sur la trace (section 8) |
| Le tag de Post s'arrete-t-il toujours ? | Post 1921 | **probleme ouvert** | comportements mesures, aucun tranche (section 7) |
| S seul est-il universel ? | Challenge 2021 | probleme ouvert ($20 000) | cadre expose, non attaque (section 9) |

Le pli 4 du Origami Wolfram demandait la reception critique "au meme rang que les theses" :
c'est la table ci-dessus -- chaque these a son statut, chaque statut a sa charge de preuve, et
le carnet ne depasse jamais la sienne.

## Vérifiez votre compréhension

1. La section 2 exécute les deux ordres de réduction sur `K I OMEGA` — quel ordre termine, et que montre celui qui ne termine pas ?
2. L'arithmétique de Church est calculée en section 5 — pourquoi `2^3` n'est-il pas symétrique à `3^2` ?
3. Church-Rosser est attesté en section 6 par 30 ordres vers une forme normale — pourquoi cela reste-t-il un témoin empirique et non une preuve ?

## Exercice 1 -- Le combinateur d'echange, en pur S/K

Le combinateur `C` echange ses deux derniers arguments : `C f x y -> f y x`. Construisez-le
par abstraction de bracket (`lam`, trois variables liees), appliquez-le a `f x y` et
verifiez que la forme normale est `f y (x)` -- c'est-a-dire `f y x` en association a gauche.

**Etapes** : (1) ecrire `C` comme `\f x y. f y x` via trois `lam` imbriques ; (2) construire
`C f x y` avec `ap` ; (3) reduire en ordre normal et afficher la forme normale ; (4) compter
les noeuds de `C` avec la fonction `taille` de la section 4 -- le prix de l'echange.

In [11]:
// Exercice 1 — Construire et appliquer C
{
    use std::rc::Rc;

    // Socle local : terme, construction, abstraction, réduction normale et affichage.
    type TR = Rc<Term>;
    enum Term {
        Var(String),
        App(TR, TR),
    }

    #[allow(dead_code)] // À réutiliser en complétant les TODO de l'exercice.
    fn var(nom: &str) -> TR {
        Rc::new(Term::Var(nom.to_owned()))
    }

    fn ap(f: TR, x: TR) -> TR {
        Rc::new(Term::App(f, x))
    }

    fn est_var(terme: &TR, nom: &str) -> bool {
        matches!(terme.as_ref(), Term::Var(v) if v == nom)
    }

    #[allow(dead_code)]
    fn contient(terme: &TR, nom: &str) -> bool {
        match terme.as_ref() {
            Term::Var(v) => v == nom,
            Term::App(f, x) => contient(f, nom) || contient(x, nom),
        }
    }

    // Abstraction générique comme à la section 4, sans optimisation B/C.
    #[allow(dead_code)]
    fn lam(x: &str, corps: &TR) -> TR {
        if !contient(corps, x) {
            return ap(var("K"), Rc::clone(corps));
        }
        match corps.as_ref() {
            Term::Var(_) => var("I"),
            Term::App(f, a) => ap(ap(var("S"), lam(x, f)), lam(x, a)),
        }
    }

    fn to_str(terme: &TR) -> String {
        match terme.as_ref() {
            Term::Var(nom) => nom.clone(),
            Term::App(f, x) => {
                let gauche = to_str(f);
                let droite = to_str(x);
                if matches!(x.as_ref(), Term::App(..)) {
                    format!("{gauche} ({droite})")
                } else {
                    format!("{gauche} {droite}")
                }
            }
        }
    }

    fn apply_root(terme: &TR) -> Option<TR> {
        let Term::App(f, x) = terme.as_ref() else { return None; };
        if let Term::App(h, g) = f.as_ref() {
            if let Term::App(s, sf) = h.as_ref() {
                if est_var(s, "S") {
                    return Some(ap(ap(Rc::clone(sf), Rc::clone(x)), ap(Rc::clone(g), Rc::clone(x))));
                }
            }
            if est_var(h, "K") {
                return Some(Rc::clone(g));
            }
        }
        est_var(f, "I").then(|| Rc::clone(x))
    }

    // Ordre normal : racine, puis fonction, puis argument.
    fn step_normal(terme: &TR) -> Option<TR> {
        if let Some(suite) = apply_root(terme) {
            return Some(suite);
        }
        let Term::App(f, x) = terme.as_ref() else { return None; };
        if let Some(suite) = step_normal(f) {
            return Some(ap(suite, Rc::clone(x)));
        }
        step_normal(x).map(|suite| ap(Rc::clone(f), suite))
    }

    fn reduce_normal(terme: &TR, borne: usize) -> (Option<TR>, usize) {
        let mut courant = Rc::clone(terme);
        for pas in 0..borne {
            let Some(suite) = step_normal(&courant) else { return (Some(courant), pas); };
            courant = suite;
        }
        (None, borne)
    }

    // TODO étudiant : construire C = \f x y. f y x par abstraction de bracket.
    // Étape 1 : utiliser successivement lam pour lier f, x et y.
    let c: Option<TR> = None;

    // Étape 2 : appliquer C à f, x, y.
    let applique: Option<TR> = None; // TODO étudiant.

    // Étapes 3-4 : réduire et mesurer.
    if let (Some(c), Some(applique)) = (&c, &applique) {
        let (nf, pas) = reduce_normal(applique, 500);
        if let Some(nf) = nf {
            println!("forme normale : {} ({pas} pas)", to_str(&nf));
        } else {
            println!("aucune forme normale atteinte ({pas} pas)");
        }
        // TODO étudiant : reprendre la fonction taille de la section 4 dans ce bloc
        // et compter les nœuds (Var ou App) du terme C, et non de son application.
        let _ = c;
    } else {
        println!("Exercice a completer : C et applique");
    }
};

Exercice a completer : C et applique


## Exercice 2 -- L'exponentiation n'est pas symetrique

La section 5 calcule `2^3 = 8` avec `exp = \m n. n m`. Verifiez que **l'ordre des operandes
compte** : `3^2` doit donner `9`, pas `8`. Construisez les deux termes, reduisez-les et
convertissez avec `church_to_int`.

**Etapes** : (1) reconstruire `exp` par `lam` ; (2) `exp trois deux` et `exp deux trois` ;
(3) convertir les deux resultats ; (4) expliquer en une phrase pourquoi `n m` donne `m^n`
et non `m^n` inverse -- que represente l'application de `n` a `m` ?

In [12]:
// Exercice 2 — Comparer les deux ordres d’opérandes
{
    use std::rc::Rc;

    // Socle local : construction, abstraction et décodage des numéraux de Church.
    type TR = Rc<Term>;
    enum Term {
        Var(String),
        App(TR, TR),
    }

    fn var(nom: &str) -> TR {
        Rc::new(Term::Var(nom.to_owned()))
    }

    fn ap(f: TR, x: TR) -> TR {
        Rc::new(Term::App(f, x))
    }

    fn est_var(terme: &TR, nom: &str) -> bool {
        matches!(terme.as_ref(), Term::Var(v) if v == nom)
    }

    fn contient(terme: &TR, nom: &str) -> bool {
        match terme.as_ref() {
            Term::Var(v) => v == nom,
            Term::App(f, x) => contient(f, nom) || contient(x, nom),
        }
    }

    // Appliquer les trois schémas de bracket sans raccourci B/C.
    fn lam(x: &str, corps: &TR) -> TR {
        if !contient(corps, x) {
            return ap(var("K"), Rc::clone(corps));
        }
        match corps.as_ref() {
            Term::Var(_) => var("I"),
            Term::App(f, a) => ap(ap(var("S"), lam(x, f)), lam(x, a)),
        }
    }

    fn church(n: usize) -> TR {
        let f = var("f");
        let mut corps = var("x");
        for _ in 0..n {
            corps = ap(Rc::clone(&f), corps);
        }
        lam("f", &lam("x", &corps))
    }

    fn apply_root(terme: &TR) -> Option<TR> {
        let Term::App(f, x) = terme.as_ref() else { return None; };
        if let Term::App(h, g) = f.as_ref() {
            if let Term::App(s, sf) = h.as_ref() {
                if est_var(s, "S") {
                    return Some(ap(ap(Rc::clone(sf), Rc::clone(x)), ap(Rc::clone(g), Rc::clone(x))));
                }
            }
            if est_var(h, "K") {
                return Some(Rc::clone(g));
            }
        }
        est_var(f, "I").then(|| Rc::clone(x))
    }

    // Ordre normal : racine, puis fonction, puis argument.
    fn step_normal(terme: &TR) -> Option<TR> {
        if let Some(suite) = apply_root(terme) {
            return Some(suite);
        }
        let Term::App(f, x) = terme.as_ref() else { return None; };
        if let Some(suite) = step_normal(f) {
            return Some(ap(suite, Rc::clone(x)));
        }
        step_normal(x).map(|suite| ap(Rc::clone(f), suite))
    }

    fn reduce_normal(mut terme: TR, borne: usize) -> Option<TR> {
        for _ in 0..borne {
            let Some(suite) = step_normal(&terme) else { return Some(terme); };
            terme = suite;
        }
        None
    }

    // Appliquer à f et x, puis lire le nombre d'applications dans la forme normale.
    fn church_to_int(terme: &TR) -> Option<usize> {
        let applique = ap(ap(Rc::clone(terme), var("f")), var("x"));
        let nf = reduce_normal(applique, 100_000)?;
        let mut courant = &nf;
        let mut entier = 0;
        while let Term::App(f, x) = courant.as_ref() {
            if !est_var(f, "f") { return None; }
            entier += 1;
            courant = x;
        }
        est_var(courant, "x").then_some(entier)
    }

    let (deux, trois) = (church(2), church(3));
    // TODO etudiant : construire \m n. n m par lam.
    let exp: Option<TR> = None;

    if let Some(exp) = exp {
        // TODO etudiant : construire le terme pour 3^2 (témoin Python : 9).
        let trois_puissance_deux: Option<TR> = None;
        // TODO etudiant : construire le terme pour 2^3 (témoin Python : 8).
        let deux_puissance_trois: Option<TR> = None;
        let _ = (&exp, &deux, &trois);

        let premier = trois_puissance_deux.as_ref().and_then(church_to_int);
        let second = deux_puissance_trois.as_ref().and_then(church_to_int);
        if let (Some(premier), Some(second)) = (premier, second) {
            println!("3 ^ 2 = {premier} | 2 ^ 3 = {second}");
        } else {
            println!("Completer les applications et verifier leur reduction");
        }
    } else {
        println!("Exercice a completer : exp");
    }
};

Exercice a completer : exp


## Exercice 3 -- Cartographier les graines du tag de Post

La section 7 teste quatre graines. Ecrivez un balayage : pour toutes les graines binaires de
longueur 4 a 8, executez `tag_evolve(&graine, 100_000)`, comptez les verdicts
`halted` / `cycle` / `unbounded`, et relevez la plus longue croissance avant cycle.

**Etapes** : (1) generer les graines avec le formatage binaire de `format!`, en conservant les zeros de tete ; (2) classer
chaque verdict ; (3) afficher le decompte par longueur ; (4) relever la graine dont le
transitoire est le plus long -- la reponse reste nettement sous 100 etats, verifiez-le ;
(5) montrer que le transitoire peut pourtant depasser 100 etats, mais hors de ce domaine :
cherchez une graine de longueur superieure a 8 (il en existe une des la longueur 12), et
verifiez que votre mesure est reproductible (le systeme est deterministe : deux executions
donnent le meme verdict).

In [13]:
// Exercice 3 — Classer les graines et mesurer les transitoires
{
    use std::collections::{BTreeMap, HashSet};

    // Socle local : un pas retire trois symboles, puis ajoute le bloc choisi par la tête.
    #[allow(dead_code)]
    fn tag_step(etat: &str) -> Option<String> {
        let suffixe = etat.get(3..)?;
        let bloc = match etat.as_bytes().first().copied()? {
            b'0' => "00",
            b'1' => "1101",
            _ => return None,
        };
        let mut suite = suffixe.to_owned();
        suite.push_str(bloc);
        Some(suite)
    }

    // Comme à la section 7, la trace inclut l'état qui ferme le cycle.
    #[allow(dead_code)]
    fn tag_evolve(graine: &str, borne: usize) -> (Vec<String>, &'static str) {
        let mut courant = graine.to_owned();
        let mut trace = vec![courant.clone()];
        let mut vus = HashSet::from([courant.clone()]);
        for _ in 0..borne {
            let Some(suite) = tag_step(&courant) else { return (trace, "halted"); };
            trace.push(suite.clone());
            if !vus.insert(suite.clone()) {
                return (trace, "cycle");
            }
            courant = suite;
        }
        (trace, "unbounded")
    }

    // TODO etudiant : {(longueur, verdict): nombre}.
    let decompte: Option<BTreeMap<(usize, &'static str), usize>> = None;
    // TODO etudiant : (graine, nombre d'etats enregistres jusqu'a la detection du cycle).
    let transitoire_max: Option<(String, usize)> = None;
    // TODO etudiant : balayer les graines et renseigner les deux valeurs.
    // L'indice de sortie conserve la notation Python ; en Rust, utiliser
    // tag_evolve(&graine, 100_000) et trace.len(). Cette longueur inclut l'état
    // initial et l'état répété : ce n'est pas la longueur du seul précycle.
    let _ = (&decompte, &transitoire_max);
    println!("Exercice a completer : cartographier les graines de longueur 4 a 8");
    println!("Indice : cb.tag_evolve(graine) renvoie (trace, verdict) ; len(trace) mesure le transitoire.");
};

Exercice a completer : cartographier les graines de longueur 4 a 8


Indice : cb.tag_evolve(graine) renvoie (trace, verdict) ; len(trace) mesure le transitoire.


## À retenir

*Carnet d'origine ICT (pli 4 de l'Origami Wolfram, anciennement ICT-51), rattaché à la série
Complexity — voir le [registre des renommages](../../../docs/reference/rename-ledger.tsv).*

*Ce carnet couvre le premier point du pli 4 : les combinateurs S/K/I comme representation
ultime du calcul -- syntaxe, reduction, strategies, arithmetique de Church -- et la borne
d'irreductibilite que le systeme de tag de Post leur oppose depuis 1921. Trois statuts y
coexistent sans se confondre : des theoremes executes (`S K K x -> x`, l'arithmetique), un
temoin empirique d'un theoreme non prouve ici (Church-Rosser), et deux problemes ouverts
(l'arret du tag de Post, l'universalite de S seul).*

Ce qui reste au pli : la **preuve formelle de la confluence** (grain B, organe Lean -- les
relations de Mathlib y seront nommees avant toute implementation), et la lecture des textes
2025 (*Ruliology of Lambdas*) que la matrice du pli signale sans les avoir encore extraits.
La lecon de methode vaut pour toute la serie : un moteur minimal n'est pas un moteur pauvre --
S et K tiennent en deux lignes, et un siecle de reception n'a pas epuise ce qu'elles contiennent.

## Pour aller plus loin

- **[Complexity-08c — Machines de Gödel](Complexity-08c-MachinesDeGoedel-Python.ipynb)** : le carnet suivant de l'accrétion, où la self-modification prouvée prolonge la question de l'irréductibilité.
- **Jumeau C#** → [Complexity-08b-CombinateursSKI-TagDePost-CSharp](Complexity-08b-CombinateursSKI-TagDePost-CSharp.ipynb) — le même carnet, porté en C#.
- **[ICT-Synthèse — CrossSubstrat](../../IIT/ICT-Series/ICT-Synthese-CrossSubstrat-Python.ipynb)** : le capstone de la série d'origine, qui mesure ce que les instruments transfèrent d'un substrat à l'autre.
- **[ICT-0 — Cadrage de la série](../../IIT/ICT-Series/ICT-0-Framing.md)** : le principe directeur de la série ICT d'origine.

### Note de parité avec le jumeau Python

| Partie | Sorties à comparer |
|---|---|
| Introduction | La ligne des règles est comparée à l'identique ; la première ligne nomme le moteur natif Rust au lieu de l'import Python. |
| Sections 1 à 5 | Traces, compteurs de réduction, abstractions, tailles et résultats des opérations de Church : comparaison exacte. |
| Section 6 | Les chemins tirés ne sont pas comparés ; la forme normale et les statistiques affichées le sont exactement. |
| Sections 7 et 8 | Verdicts, résumés de traces, blocs absents, mots de substitution et décomptes : comparaison exacte. |
| Exercices | Les messages des stubs sont comparés à l'identique, y compris l'indice en notation Python de l'exercice 3. |